# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution
This notebook provides a template and executable steps for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using the [Croissant schema](https://mlcommons.org/croissant/) and is published at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Dataset description: {metadata.description}\n")
print(f"Dataset identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets and their entities. All entities are referenced by their `@id` fields as required by the Croissant specification.

In [ ]:
# List available RecordSets and their Fields, referencing by @id
record_sets = metadata.recordSet
if not record_sets:
    # Try to infer the list from schema (older spec or different location)
    record_sets = getattr(metadata, 'record_sets', [])

print("Available RecordSets:")
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
    rs_name = rs.get('name') if isinstance(rs, dict) and 'name' in rs else getattr(rs, 'name', None)
    print(f"- @id: {rs_id} | Name: {rs_name}")
    # List its fields if available
    fields = rs.get('field') if isinstance(rs, dict) and 'field' in rs else getattr(rs, 'field', [])
    if fields:
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            f_id = field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, '@id', None)
            f_name = field.get('name') if isinstance(field, dict) and 'name' in field else getattr(field, 'name', None)
            print(f"    - Field @id: {f_id} | Name: {f_name}")

### Example records from a RecordSet
Let's display a few records (rows) from one of the available RecordSets by its `@id`.
*(Replace `<record_set_id>` with a real `@id` from the output above for detailed exploration.)*

In [ ]:
# Choose a RecordSet @id to preview (update if necessary based on previous output)
# Example: record_set_id = 'http://mlcommons.org/croissant/recordSet/main'
if record_sets:
    # Automatically choose the first record set for demonstration
    chosen_rs = record_sets[0]
    record_set_id = chosen_rs['@id'] if isinstance(chosen_rs, dict) and '@id' in chosen_rs else getattr(chosen_rs, '@id', None)

    print(f"\nShowing first 3 records from RecordSet {record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        if i>=3: break
        print(record)
else:
    print("No RecordSets found in the metadata.")

## 3. Data Extraction
Load all records from the chosen RecordSet into a pandas DataFrame for analysis. All data access uses the `@id` of each RecordSet.

In [ ]:
dataframes = {}
list_of_record_sets_ids = []

# Get the @id for each RecordSet in the dataset
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
    list_of_record_sets_ids.append(rs_id)

# Load data for each RecordSet as DataFrame
for rs_id in list_of_record_sets_ids:
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Preview columns and head for the first RecordSet
main_record_set_id = list_of_record_sets_ids[0] if list_of_record_sets_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print("\nColumn names:")
    print(dataframes[main_record_set_id].columns.tolist())
    print("\nSample rows:")
    display(dataframes[main_record_set_id].head())
else:
    print("No RecordSets or records loaded.")

## 4. Exploratory Data Analysis (EDA)
We will filter, normalize, and group records using one numeric field and one grouping field as examples.

All field and column references always use their `@id` to avoid ambiguity and follow best Croissant practices.

*Update the `numeric_field_id` and `group_field_id` variables below as appropriate for the main table based on the real dataset schema. Below, we attempt to select candidates automatically, but you may override based on data overview.*

In [ ]:
# Identify a numeric field and a group field by @id, or update them as needed based on your data overview.
import numpy as np

df = dataframes[main_record_set_id]

# Try to auto-detect a numeric column (e.g., demography: age or diagnostic days)
numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[col])]
numeric_field_id = numeric_candidates[0] if numeric_candidates else df.columns[0]  # Default to first if uncertain

# Try to auto-detect a grouping/categorical column
group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and df[col].nunique()<10]
group_field_id = group_candidates[0] if group_candidates else df.columns[0]

print(f"Selected numeric field for analysis: {numeric_field_id}")
print(f"Selected group field for analysis: {group_field_id}\n")

# Filter: Show rows where the numeric field > mean + std (outlier high values)
threshold = df[numeric_field_id].mean() + df[numeric_field_id].std()
filtered_df = df[df[numeric_field_id] > threshold].copy()

print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric column for the filtered subset
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (
    filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1
)

print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally: Group by categorical field and show means
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nMean {numeric_field_id} for each {group_field_id} (filtered):")
    display(grouped_df)
else:
    print(f"\nGroup field {group_field_id} not found in columns.")

## 5. Visualization
Let's plot the main numeric field's distribution and compare between groups (if group field is detected).

We use seaborn/matplotlib to display these relationships based on the DataFrame, always referencing columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.histplot(df[numeric_field_id], bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

if group_field_id in df.columns:
    plt.figure(figsize=(12,6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded clinical and molecular data from a Croissant schema using the `mlcroissant` library,
- Explored metadata, record sets, and fields using their unique `@id` references,
- Loaded the main table into a DataFrame, performed basic filtering, normalization and grouping operations,
- Visualized core features to support further biomarker and clinical analysis.

This workflow can be adapted to any Croissant-enabled dataset. Please update or extend this notebook with domain-specific analyses and hypotheses as needed.

For more details about the FAIR² Croissant schema and `mlcroissant`, see:
- Dataset publication: https://sen.science/doi/10.71728/senscience.qs2f-h81p
- mlcroissant docs: https://github.com/mlcommons/croissant